# Credit Card Fraud Detection Using Machine Learning and SMOTE


## Objectives
- Explore imbalanced fraud dataset
- Perform EDA and visualize class imbalance
- Apply Undersampling, Oversampling, and SMOTE
- Train Random Forest and XGBoost models
- Evaluate using Precision, Recall, and ROC-AUC
- Discuss business decision thresholds


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
try:
    from xgboost import XGBClassifier
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier

    class XGBClassifier:
        """Fallback wrapper using HistGradientBoostingClassifier when xgboost is unavailable."""
        def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=None, random_state=None, **kwargs):
            self.model = HistGradientBoostingClassifier(
                max_iter=n_estimators,
                learning_rate=learning_rate,
                random_state=random_state
            )

        def fit(self, X, y, **kwargs):
            return self.model.fit(X, y, **kwargs)

        def predict(self, X):
            return self.model.predict(X)

        def predict_proba(self, X):
            return self.model.predict_proba(X)

        def set_params(self, **params):
            self.model.set_params(**params)
            return self

sns.set_style("whitegrid")


ModuleNotFoundError: No module named 'xgboost'

In [ ]:
df = pd.read_csv('creditcard.csv')
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df['Class'].value_counts()

In [ ]:
sns.countplot(x='Class', data=df)
plt.title('Class Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_pred))

In [ ]:
undersample = RandomUnderSampler(random_state=42)
X_under, y_under = undersample.fit_resample(X_train, y_train)

oversample = RandomOverSampler(random_state=42)
X_over, y_over = oversample.fit_resample(X_train, y_train)

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

print(y_smote.value_counts())

In [ ]:
sns.countplot(x=y_smote)
plt.title('Balanced Dataset using SMOTE')
plt.show()

In [ ]:
rf_smote = RandomForestClassifier(n_estimators=200, random_state=42)
rf_smote.fit(X_smote, y_smote)

rf_pred = rf_smote.predict(X_test)

print(classification_report(y_test, rf_pred))
print('ROC-AUC:', roc_auc_score(y_test, rf_pred))

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

xgb.fit(X_smote, y_smote)
xgb_pred = xgb.predict(X_test)

print(classification_report(y_test, xgb_pred))
print('ROC-AUC:', roc_auc_score(y_test, xgb_pred))

In [ ]:
rf_probs = rf_smote.predict_proba(X_test)[:,1]

fpr, tpr, thresholds = roc_curve(y_test, rf_probs)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label='Random Forest')
plt.plot([0,1],[0,1],'--')
plt.legend()
plt.title('ROC Curve')
plt.show()

In [ ]:
cm = confusion_matrix(y_test, rf_pred)

sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
precision = precision_score(y_test, rf_pred)
recall = recall_score(y_test, rf_pred)

print('Precision:', precision)
print('Recall:', recall)

for t in [0.3,0.4,0.5,0.6,0.7]:
    pred = (rf_probs >= t).astype(int)
    p = precision_score(y_test, pred)
    r = recall_score(y_test, pred)
    print(f'Threshold={t} Precision={p:.3f} Recall={r:.3f}')